ADDING SOME LINKS \
https://www.youtube.com/watch?v=uljYTx7nzy8

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

from typing import List, NamedTuple, Optional, Tuple, Dict, Any
from functools import reduce

from abc import ABC, abstractmethod

#spark = SparkSession.builder.appName("OTIF").getOrCreate()

# ###############################################################################
# # Unity Catalog #
# ###############################################################################

# https://blog.stephane-robert.info/docs/developper/programmation/python/dataclasses/
class UnityCatalog(NamedTuple):
    catalog: str = "sandbox_dev"
    schema: str = "supplychain_explo"
    volume : str = "sc1"
    folder : str = "otif"

uc = UnityCatalog()

### ## # **WITHOUT OOP & WITHOUT DESIGN PATTERNS**

In [0]:
# TO DO : 
# https://medium.com/@lucianosantosg3/medallion-architecture-pipeline-in-an-oop-structure-d3290ace29b4

In [2]:
class BronzeIngestion:
    def __init__(self) -> None:
        self.spark = SparkSession.builder.appName("OTIF").getOrCreate()

    # def __init__(self, catalog, schema, volume, folder, fily, spark, formatt, sep):
    #     self.catalog = catalog
    #     self.schema = schema
    #     self.volume = volume
    #     self.folder = folder
    #     self.fily = fily
    #     self.spark = spark
    #     self.formatt = formatt
    #     self.sep = sep

    def spark_read_csv(self, catalog: str, schema :str, volume: str, 
                             folder: str, fily: str, spark=spark, 
                             formatt: str = 'csv', sep: str = ";") -> DataFrame:
        
                 fily = fily + ".csv"

                 df = spark.read.format(formatt) \
                .option("header", "true") \
                .option('delimiter', sep) \
                .load(f"/Volumes/{catalog}/{schema}/{volume}/{folder}/{fily}")

                 return df

In [0]:

class SilverLayer1:
    def __init__(self) -> None:
        pass

    # Otif DF Columns Names Dict
    def build_colotif(self, df : DataFrame, lst : list()) -> DataFrame:
                       lstret = [F.col(column).alias(col) for column, col in zip(df.columns, lst)]
                       return df.select(lstret)

    # Lowercase Columns
    def build_lowercase(self, df: DataFrame) -> DataFrame :
                        lowered_columns = [F.col(column).alias(column.lower()) for column in df.columns]
                        return df.select(lowered_columns)

    # Remove Blanks
    def build_woblk(self, df: DataFrame) -> DataFrame :
                         woblk = [F.col(column).alias(column.replace(" ", "")) for column in df.columns]
                         return df.select(woblk)
                     
    # Remove Unwanted Columns In MD
    def build_mdemo(self, df: DataFrame) -> DataFrame : 
                          cols2Keep = [col for col in df.columns if not col.startswith('_c1')]
                          return df.select(cols2Keep)

In [0]:
class SilverLayer2:
    def __init__(self) -> None:
        pass
    
    def build_join(self, dfa:DataFrame, dfb:DataFrame, cola: list[str] | str , colb: list[str] | str, typOfJoin: str) -> DataFrame :
        a = dfa    
        b = dfb
        if isinstance(cola, str):
            c = a.join(b, \
                  a[cola] == b[colb],\
                  how = typOfJoin )
    
        if isinstance(cola, list):
            c = a.join(b, \
                  on = [a[cola[0]] == b[colb[0]],
                      a[cola[1]] == b[colb[1]]],\
                  how = typOfJoin )
    
        return c

    def build_otif_cast(self, df: DataFrame) -> DataFrame:
        return df.withColumns({'impactedmonth' : F.to_date(regexp_replace('impactedmonth', "/", "-"),"dd-mm-yyyy") \
                           ,'deliverydate' : F.to_date(regexp_replace('deliverydate', "/", "-"),"dd-mm-yyyy") \
                           ,'targeteddate' : F.to_date(regexp_replace('targeteddate', "/", "-"), "dd-mm-yyyy") \
                           ,'poquantity' : to_number(regexp_replace('poquantity', ",", "."), lit("9999999.9999"))                         
                           } )
    
    def build_otif_gap_threshold(self, df: DataFrame) -> DataFrame:
        return df.withColumns({'gapinbusdays': F.expr(
                                       """
                                       CASE WHEN targeteddate > deliverydate THEN datediff(deliverydate, targeteddate)
                                            WHEN targeteddate <= deliverydate THEN size(filter(sequence(targeteddate, deliverydate, interval 1 day), x -> date_format(x, 'E') not in ('Sat', 'Sun')))
                                       END
                                        """) \
                           ,'qtythreshold' :  0.05*F.col('poquantity')                           
                           } )
    
    def build_otif_recalculations(self, df: DataFrame) -> DataFrame:
        return df.withColumns({ 'OTIF_NonOTIF_ReCalc' : when(F.lower(col('plannerotifcorr')).startswith('y'), 'OTIF') \
                            .when(F.lower(col('plannerotifcorr')).startswith('n'), 'NONOTIF') \
                            .when((col('gapinbusdays') <= col('leadtime') + 7) & (col('qtythreshold') >= col('remainingquantities')), 'OTIF') \
                            .otherwise(F.lit('NONOTIF'))                 
                           } )

In [0]:
class BronzeIngestion:
    def __init__(self) -> None:
        self.spark = SparkSession.builder.appName("OTIF").getOrCreate()

    # def __init__(self, catalog, schema, volume, folder, fily, spark, formatt, sep):
    #     self.catalog = catalog
    #     self.schema = schema
    #     self.volume = volume
    #     self.folder = folder
    #     self.fily = fily
    #     self.spark = spark
    #     self.formatt = formatt
    #     self.sep = sep

    def spark_read_csv(self, catalog: str, schema :str, volume: str, 
                             folder: str, fily: str, spark=spark, 
                             formatt: str = 'csv', sep: str = ";") -> DataFrame:
        
                 fily = fily + ".csv"

                 df = spark.read.format(formatt) \
                .option("header", "true") \
                .option('delimiter', sep) \
                .load(f"/Volumes/{catalog}/{schema}/{volume}/{folder}/{fily}")

                 return df


class SilverService2(ABC):
    @abstractmethod
    def transform(self, data):
        pass

class SilverLayer1(SilverService2):
    def __init__(self) -> None:
        pass

    def transform(self, data):
        return data

    # Otif DF Columns Names Dict
    def build_colotif(self, df : DataFrame, lst : list()) -> DataFrame:
                       lstret = [F.col(column).alias(col) for column, col in zip(df.columns, lst)]
                       return df.select(lstret)

    # Lowercase Columns
    def build_lowercase(self, df: DataFrame) -> DataFrame :
                        lowered_columns = [F.col(column).alias(column.lower()) for column in df.columns]
                        return df.select(lowered_columns)

    # Remove Blanks
    def build_woblk(self, df: DataFrame) -> DataFrame :
                         woblk = [F.col(column).alias(column.replace(" ", "")) for column in df.columns]
                         return df.select(woblk)
                     
    # Remove Unwanted Columns In MD
    def build_mdemo(self, df: DataFrame) -> DataFrame : 
                          cols2Keep = [col for col in df.columns if not col.startswith('_c1')]
                          return df.select(cols2Keep)

### ## # **WITH OOP & DESIGN PATTERNS**

In [0]:
# V1 
class SilverService2(ABC):
    def __init__(self) -> None:
        pass

    @abstractmethod
    def transform(self, data):
        pass

class SilverLayer2(SilverService2):
    def __init__(self) -> None:
        pass
        
    def transform(self, data):
        return data  
    
    def build_join(self, dfa:DataFrame, dfb:DataFrame, cola: list[str] | str , colb: list[str] | str, typOfJoin: str) -> DataFrame :
        a = dfa    
        b = dfb
        if isinstance(cola, str):
            c = a.join(b, \
                  a[cola] == b[colb],\
                  how = typOfJoin )
    
        if isinstance(cola, list):
            c = a.join(b, \
                  on = [a[cola[0]] == b[colb[0]],
                      a[cola[1]] == b[colb[1]]],\
                  how = typOfJoin )
    
        return c

    def build_otif_cast(self, df: DataFrame) -> DataFrame:
        return df.withColumns({'impactedmonth' : F.to_date(regexp_replace('impactedmonth', "/", "-"),"dd-mm-yyyy") \
                           ,'deliverydate' : F.to_date(regexp_replace('deliverydate', "/", "-"),"dd-mm-yyyy") \
                           ,'targeteddate' : F.to_date(regexp_replace('targeteddate', "/", "-"), "dd-mm-yyyy") \
                           ,'poquantity' : to_number(regexp_replace('poquantity', ",", "."), lit("9999999.9999"))                         
                           } )
    
    def build_otif_gap_threshold(self, df: DataFrame) -> DataFrame:
        return df.withColumns({'gapinbusdays': F.expr(
                                       """
                                       CASE WHEN targeteddate > deliverydate THEN datediff(deliverydate, targeteddate)
                                            WHEN targeteddate <= deliverydate THEN size(filter(sequence(targeteddate, deliverydate, interval 1 day), x -> date_format(x, 'E') not in ('Sat', 'Sun')))
                                       END
                                        """) \
                           ,'qtythreshold' :  0.05*F.col('poquantity')                           
                           } )
    
    def build_otif_recalculations(self, df: DataFrame) -> DataFrame:
        return df.withColumns({ 'OTIF_NonOTIF_ReCalc' : when(F.lower(col('plannerotifcorr')).startswith('y'), 'OTIF') \
                            .when(F.lower(col('plannerotifcorr')).startswith('n'), 'NONOTIF') \
                            .when((col('gapinbusdays') <= col('leadtime') + 7) & (col('qtythreshold') >= col('remainingquantities')), 'OTIF') \
                            .otherwise(F.lit('NONOTIF'))                 
                           } )

PRENDRE CELLE CI-DESSOUS

In [3]:
# V2
class SilverService3(ABC):
    def __init__(self, next = None):
        self.next = next
    
    def set_next(self, next):
        self.next = next
        return next

    @abstractmethod
    def transform(self):
        pass

class BuildJoined(SilverService3):    
    def __init__(self, dfa:DataFrame, dfb:DataFrame, cola: list[str] | str , colb: list[str] | str, typOfJoin: str):
        self.dfa = dfa
        self.dfb = dfb
        self.cola = cola
        self.colb = colb
        self.typOfJoin = typOfJoin

    def transform(self) -> DataFrame :

        if isinstance(self.cola, str):
            dfc = self.dfa.join(self.dfb, \
                  self.dfa[self.cola] == self.dfb[self.colb],\
                  how = self.typOfJoin)
            dfc = dfc.drop(self.colb)
    
        if isinstance(self.cola, list):
            dfc = self.dfa.join(self.dfb, \
                  on = [self.dfa[self.cola[0]] == self.dfb[self.colb[0]],
                      self.dfa[self.cola[1]] == self.dfb[self.colb[1]]],\
                  how = self.typOfJoin )
            dfc = dfc.drop(*self.colb)
    
        return dfc
    
class BuildColotif(SilverService3):
    def __init__(self, df : DataFrame, lst : list[str]):
        self.df = df
        self.lst = lst

    def transform(self) -> DataFrame:
                       lstret = [F.col(column).alias(col) for column, col in zip(self.df.columns, self.lst)]
                       return self.df.select(lstret)

    # Lowercase Columns
class BuildLowercase(SilverService3):
    def __init__(self, df : DataFrame) : 
        self.df = df  

    def transform(self) -> DataFrame :
                        lowered_columns = [F.col(column).alias(column.lower()) for column in self.df.columns]
                        return self.df.select(lowered_columns)

    # Trim blanks
class BuildTrimBlanks(SilverService3):
    def __init__(self, df : DataFrame) :
        self.df = df

    def transform(self) -> DataFrame : 
                woblk = [F.col(col).alias(col.strip()) for col in self.df.columns]
                
                return self.df.select(woblk)

    # Replace Blanks
class BuildReplaceBlank(SilverService3):
    def __init__(self, df : DataFrame) : 
        self.df = df

    def transform(self) -> DataFrame :
                      woblk = [F.col(column).alias(column.replace(" ", "_")) for column in self.df.columns]

                      return self.df.select(woblk)
                  
    # Add Suffix To MD
class BuildAddSuffix(SilverService3):
    def __init__(self, df : DataFrame, suffix : str) : 
        self.df = df
        self.suffix = suffix

    def transform(self) -> DataFrame :
                      #lstsfx = [F.col(column).alias(column + str(suffix)) for column in self.df.columns]
                       for column in self.df.columns:

                               self.df = self.df.withColumnRenamed(column, '{}{}'.format(column, self.suffix))

                       return self.df

    # Remove Unwanted Columns In MD
class BuildRemoveCols(SilverService3):
    def __init__(self, df : DataFrame) : 
          self.df = df

    def transform(self) -> DataFrame :
                          cols2Keep = [col for col in self.df.columns if not col.startswith('_c1')]
                          return self.df.select(cols2Keep)
                      
class BuildOtifCast(SilverService3):
    def __init__(self, df: DataFrame):
         self.df = df 

    def transform(self) -> DataFrame:
        return self.df.withColumn('impacted_month' , F.to_date('impacted_month',"dd/MM/yyyy")) \
                           .withColumn('delivery_date' , F.to_date(regexp_replace(F.col('delivery_date'), "/", "-"),"dd-MM-yyyy")) \
                           .withColumn('targeted_date' , F.to_date(regexp_replace('targeted_date', "/", "-"), "dd-MM-yyyy")) \
                           .withColumn('po_quantity' , to_number(regexp_replace('po_quantity', ",", "."), lit("9999999.9999"))                         
                            )

class BuildOtifGapThreshold(SilverService3):
    def __init__(self, df: DataFrame):
         self.df = df 

    def transform(self) -> DataFrame:
        return self.df.withColumns({'gap_in_bus_days': F.expr(
                                       """
                                       CASE WHEN targeted_date > delivery_date THEN datediff(delivery_date, targeted_date)
                                            WHEN targeted_date <= delivery_date THEN size(filter(sequence(targeted_date, delivery_date, interval 1 day), x -> date_format(x, 'E') not in ('Sat', 'Sun')))
                                       END
                                        """) \
                           ,'qty_threshold' :  0.05*F.col('po_quantity')                           
                           } )

class BuildOtifRecalculations(SilverService3):
    def __init__(self, df: DataFrame):
         self.df = df 

    def transform(self) -> DataFrame:
        return self.df.withColumns({ 'otif_nonotif_recalc' : when(F.lower(col('planner_otif_corr')).startswith('y'), 'OTIF') \
                            .when(F.lower(col('planner_otif_corr')).startswith('n'), 'NONOTIF') \
                            .when((col('gap_in_bus_days') <= col('lead_time') + 7) & (col('qty_threshold') >= col('remaining_quantities')), 'OTIF') \
                            .otherwise(F.lit('NONOTIF')) \
                                , 'year_month': to_date(concat(date_format('impacted_month', format = "yyyy-MM"), lit("-01")))              
                                , 'otif_nonotif_def' : when((F.year('impacted_month') > 2025), col('otif_nonotif_recalc')).otherwise('otif_nonotif')
                           } )


In [4]:
bronze = BronzeIngestion()

In [5]:
# MDs #
mdemo = bronze.spark_read_csv(uc.catalog, uc.schema, uc.volume, uc.folder, "mdemo")
matrix = bronze.spark_read_csv(uc.catalog, uc.schema, uc.volume, uc.folder, "matrix")

mdemo = BuildRemoveCols(mdemo).transform()
mdemo = BuildAddSuffix(mdemo, "_MD").transform()
mdemo = BuildJoined(mdemo, matrix, ['CMO_MD', 'TYPE_MD'], ["CMO", "Manufacturig  step"], 'left').transform()
mdemo = BuildLowercase(mdemo).transform()
mdemo = BuildTrimBlanks(mdemo).transform()
mdemo = BuildReplaceBlank(mdemo).transform()


In [6]:
# OTIF #
otif = bronze.spark_read_csv(uc.catalog, uc.schema, uc.volume, uc.folder, "otifdata")

otifcolsdest = ['Supplier', 'SKUs', 'Product description', 'Impacted month', 'PO Quantity', 'Remaining quantities', 'Order Num', 'Targeted date', 'Delivery date', 'Gap in Days', 'OTIF NonOTIF', 'Planner OTIF corr ', 'Primary root cause', 'Secondary root cause', 'Short problem desc', 'Corrective action proposed', 'Corrective action imp date']

otif = BuildColotif(otif, otifcolsdest).transform()
otif = BuildLowercase(otif).transform()
otif = BuildTrimBlanks(otif).transform()
otif = BuildReplaceBlank(otif).transform()
otif = BuildJoined(otif, mdemo, 'skus', 'sku_md', 'left').transform()
otif = BuildOtifCast(otif).transform()
otif = BuildOtifGapThreshold(otif).transform()
otif = BuildOtifRecalculations(otif).transform()

# **GOLD LAYER**

In [0]:
# https://stackoverflow.com/questions/42776610/from-pandas-groupby-to-pyspark-groupby

In [7]:
class GoldService(ABC):
    def __init__(self, next = None):
        self.next = next
    
    def set_next(self, next):
        self.next = next
        return next

    @abstractmethod
    def transform(self):
        pass

class BuildGrouped2Aggs(GoldService):    
      # querytype == 0 -> not query at all
      # querytype == 1 -> one query at least

    def __init__(self, dfin : DataFrame, querytype : int , queryexpr : None | str, cols2group : list[str], aggexpr : str):
        self.dfin = dfin
        self.querytype = querytype
        self.queryexpr = queryexpr
        self.cols2group = cols2group
        self.aggexpr = aggexpr

    def transform(self) -> DataFrame :

        if self.querytype == 0 :
           dfout = self.dfin.groupby(self.cols2group).agg(*eval(self.aggexpr))
        
        if self.querytype == 1 :
           dfout = self.dfin.filter(eval(self.queryexpr)).groupby(self.cols2group).agg(*eval(self.aggexpr))
    
        return dfout
    
class BuildGrouped2AggsForEmpty(GoldService):    

    def __init__(self, dfin : DataFrame, queryexpr : str, colsname : list[str], cols2group : list[str], aggexpr : str):
        self.dfin = dfin
        self.queryexpr = queryexpr
        self.cols2group = cols2group
        self.aggexpr = aggexpr
        self.colsname = colsname

    def transform(self) -> DataFrame :

        if self.dfin.filter(eval(self.queryexpr)).isEmpty():
              a = list(otif.select('year_month').distinct().collect())
              aa = [v[0] for v in a]
              b = [0]*len(aa)
              dfout = spark.createDataFrame(list(zip(aa, b)), self.colsname)
        else:
              dfout = self.dfin.filter(eval(self.queryexpr)).groupby(self.cols2group).agg(eval(self.aggexpr))
    
        return dfout
    
class BuildReduced(GoldService):

        def __init__(self, dflist : list[DataFrame], cols: list[str], typOfJoin: str):
            self.dflist = dflist
            self.cols = cols
            self.typOfJoin = typOfJoin

        def transform(self) -> DataFrame :

            dfout = reduce(lambda left, right : left.join(right, how = self.typOfJoin, on = self.cols), self.dflist)
            
            return dfout
        
class BuildFillNas(GoldService):
    
     def __init__(self, dfin : DataFrame ):
         self.dfin = dfin

     def transform(self) -> DataFrame :
         
         dfout = self.dfin.fillna( 0, subset=["missed_pos"] )
         return dfout
        
class BuildAddOtif(GoldService):

        def __init__(self, dfin : DataFrame):
            self.dfin = dfin
        
        def transform(self) -> DataFrame :

            dfout = self.dfin.withColumn( 'otif', (F.col("missed_pos") / F.col("pos"))*100 )
            return dfout
        
class BuildDropCol(GoldService):

        def __init__(self, dfin : DataFrame, colo : str):
            self.dfin = dfin
            self.colo = colo
        
        def transform(self) -> DataFrame :

            dfout = self.dfin.drop(self.colo)
            return dfout
        
class BuildAddColPos(GoldService):

        def __init__(self, dfin : DataFrame, pos : int, colo : str, valuecolo : str):
            self.dfin = dfin
            self.pos = pos
            self.colo = colo
            self.valuecolo = valuecolo

        def transform(self) -> DataFrame :
            cols = self.dfin.columns
            dfout = self.dfin.select(
                                      *cols[:self.pos] \
                                      ,F.lit(self.valuecolo).alias(self.colo) \
                                      ,*cols[self.pos:]
                                     )
            return dfout
        
class BuildQueryTJ(GoldService):

        def __init__(self, dfin : DataFrame):
            self.dfin = dfin
        
        def transform(self) -> DataFrame :

            dfout = self.dfin.filter( col('cmo_md') != 'TJ')
            return dfout

In [8]:
# Step 1 #
gypolist1 = BuildGrouped2Aggs(otif, 0, None, ['cmo_md', 'planner_md', 'brand_md', 'year_month'], "count('skus').alias('pos'), sum('po_quantity').alias('req_qty')").transform()
gypolistALL1 = BuildGrouped2Aggs(otif, 0, None, ['year_month'], "count('skus').alias('pos'), sum('po_quantity').alias('req_qty')").transform()
gypolistTJ1 = BuildGrouped2Aggs(otif, 1, "col('cmo_md') == 'Tjoapack'", ['year_month'], "count('skus').alias('pos'), sum('po_quantity').alias('req_qty')").transform()

In [9]:
# Step 2 #
gypolist2 = BuildGrouped2Aggs(otif, 1, "col('otif_nonotif_def') == 'NONOTIF'", ['cmo_md', 'planner_md', 'brand_md', 'year_month'], "[count('otif_nonotif_def').alias('missed_pos')]").transform()
gypolistALL2 = BuildGrouped2Aggs(otif, 1, "col('otif_nonotif_def') == 'NONOTIF'", ['year_month'], "[count('otif_nonotif_def').alias('missed_pos')]").transform()
gypolistTJ2 = BuildGrouped2AggsForEmpty(otif, "(col('cmo_md') == 'Tjoapack') & (col('otif_nonotif_def') == 'NONOTIF')", 
                                              ['year_month', 'missed_pos'],
                                              ['year_month'], 
                                              "count('otif_nonotif_def').alias('missed_pos')"
                                              ).transform()

In [10]:
# Step 3 #
gypolist3 = BuildGrouped2Aggs(otif, 1, "col('otif_nonotif_def') != 'NA_SKU'", ['cmo_md', 'planner_md', 'brand_md', 'year_month'], "[sum('remaining_quantities').alias('backorders')]").transform()
gypolistALL3 = BuildGrouped2Aggs(otif, 1, "col('otif_nonotif_def') != 'NA_SKU'", ['year_month'], "[sum('remaining_quantities').alias('backorders')]").transform()
gypolistTJ3 = BuildGrouped2AggsForEmpty(otif, "(col('cmo_md') == 'Tjoapack') & (col('otif_nonotif_def') != 'NA_SKU')", 
                                              ['year_month', 'backorders'],
                                              ['year_month'], 
                                              "sum('remaining_quantities').alias('backorders')"
                                              ).transform()

In [11]:
gypo1 = [gypolist1, gypolist2, gypolist3]
gypo2 = [gypolistALL1, gypolistALL2, gypolistALL3]
gypo3 = [gypolistTJ1, gypolistTJ2, gypolistTJ3]

gypolist = BuildReduced(gypo1, ['cmo_md', 'planner_md', 'brand_md','year_month'],'left').transform()
gypolistALL = BuildReduced(gypo2, ['year_month'],'left').transform()
gypolistTJ = BuildReduced(gypo3, ['year_month'],'left').transform()

In [12]:
gypolist = BuildFillNas(gypolist).transform()
gypolistALL = BuildFillNas(gypolistALL).transform()
gypolistTJ = BuildFillNas(gypolistTJ).transform()

gypolist = BuildAddOtif(gypolist).transform()
gypolistALL = BuildAddOtif(gypolistALL).transform()
gypolistTJ = BuildAddOtif(gypolistTJ).transform()

gypolist = BuildDropCol(gypolist, 'brand_md').transform()
gypolist = BuildQueryTJ(gypolist).transform()

gypolistTJ = BuildAddColPos(gypolistTJ, 0, 'cmo_md', 'Tjoapack').transform()
gypolistTJ = BuildAddColPos(gypolistTJ, 1, 'planner_md', 'All Tjoapack').transform()

In [13]:
gypolistALL = BuildAddColPos(gypolistALL, 0, 'cmo_md', 'All').transform()
gypolistALL = BuildAddColPos(gypolistALL, 1, 'planner_md', 'All').transform()

In [0]:
# STEP NB 1 : 

gypolist1 = otif.groupby(['cmo_md', 'planner_md', 'brand_md', 'year_month']).agg(count("skus").alias("pos"),
                                                                        sum("po_quantity").alias("req_qty"))

gypolistALL1 = otif.groupby(['year_month']).agg(count("skus").alias("pos"),
                                                                        sum("po_quantity").alias("req_qty"))

gypolistTJ1 = otif.filter(col('cmo_md') == "Tjoapack").groupby(['year_month']).agg(count("skus").alias("pos"),
                                                                        sum("po_quantity").alias("req_qty"))

In [0]:
display(gypolistALL2)

cmo_md,planner_md,brand_md,year_month,missed_pos
Rottendorf,Guillaume,CABOMETYX,2026-02-01,6
Tjoapack,Guillaume,CABOMETYX,2026-07-01,1
Patheon CA,Guillaume,CABOMETYX,2026-04-01,3
Tjoapack,Guillaume,CABOMETYX,2026-06-01,2
Patheon CA,Guillaume,CABOMETYX,2026-03-01,1
Patheon CA,Guillaume,CABOMETYX,2026-02-01,7
Quotient,Gemma,SOHONOS,2026-01-01,2
Tjoapack,Guillaume,CABOMETYX,2026-05-01,2
PCI,Gemma,SOHONOS,2026-04-01,6
PCI,Gemma,SOHONOS,2026-07-01,1


In [0]:
# STEP NB 2 :

gypolist2 = otif.filter(col('otif_nonotif_def') == "NONOTIF").groupby(['cmo_md', 'planner_md', 'brand_md', 'year_month']).agg(count("otif_nonotif_def").alias("missed_pos"))

gypolistALL2 = otif.filter(col('otif_nonotif_def') == "NONOTIF").groupby(['year_month']).agg(count("otif_nonotif_def").alias("missed_pos"))

if otif.filter((col('cmo_md') == "Tjoapack") & (col('otif_nonotif_def') == "NONOTIF")).isEmpty():
    a = list(otif.select('year_month').distinct().collect())
    aa = [v[0] for v in a]
    b = [0]*len(aa)

    gypolistTJ2 = spark.createDataFrame(list(zip(aa, b)), ['year_month', 'missed_pos'])
else:
    gypolistTJ2 = otif.filter((col('cmo_md') == "Tjoapack") & (col('otif_nonotif_def') == "NONOTIF")).groupby(['year_month']).agg(count("otif_nonotif_def").alias("missed_pos"))

In [0]:
# STEP NB 3 :

gypolist3 = otif.filter(col('otif_nonotif_def') != "NA_SKU").groupby(['cmo_md', 'planner_md', 'brand_md', 'year_month']).agg(sum("remaining_quantities").alias("backorders"))

gypolistALL3 = otif.filter(col('otif_nonotif_def') != "NA_SKU").groupby(['year_month']).agg(sum("remaining_quantities").alias("backorders"))

if otif.filter((col('cmo_md') == "Tjoapack") & (col('otif_nonotif_def') != "NA_SKU")).isEmpty():
    a = list(otif.select('year_month').distinct().collect())
    aa = [v[0] for v in a]
    b = [0]*len(aa)

    gypolistTJ3 = spark.createDataFrame(list(zip(aa, b)), ['year_month', 'backorders'])
else:
    gypolistTJ3 = otif.filter((col('cmo_md') == "Tjoapack") & (col('otif_nonotif_def') != "NA_SKU")).groupby(['year_month']).agg(sum("remaining_quantities").alias("backorders"))

In [0]:
gypo1 = [gypolist1, gypolist2, gypolist3]
gypo2 = [gypolistALL1, gypolistALL2, gypolistALL3]
gypo3 = [gypolistTJ1, gypolistTJ2, gypolistTJ3]

gypolist = reduce(lambda  left,right: left.join(right, how = 'left', on = ['cmo_md', 'planner_md', 'brand_md','year_month']), gypo1)
gypolistALL = reduce(lambda  left,right: left.join(right, how='left', on='year_month'), gypo2)
gypolistTJ = reduce(lambda  left,right: left.join(right, how='left', on='year_month'), gypo3)

In [0]:
print((gypolist1.count(), len(gypolist1.columns)))
print((gypolist2.count(), len(gypolist2.columns)))
print((gypolist3.count(), len(gypolist3.columns)))
print((gypolist.count(), len(gypolist.columns)))

(170, 6)
(17, 5)
(170, 5)
(170, 8)


In [14]:
print(gypolistTJ.count(), len(gypolistTJ.columns))

19 8


In [15]:
gypolistTJ

,cmo_md,planner_md,year_month,pos,req_qty,missed_pos,backorders,otif
0,Tjoapack,All Tjoapack,2025-08-01,23,9255.0000,0,10.0,0.000000
1,Tjoapack,All Tjoapack,2026-03-01,34,7541.0000,0,50.0,0.000000
2,Tjoapack,All Tjoapack,2026-06-01,32,7077.0000,2,1050.0,6.250000
3,Tjoapack,All Tjoapack,2025-04-01,17,6654.0000,0,231.0,0.000000
4,Tjoapack,All Tjoapack,2025-11-01,23,5145.0000,0,81.0,0.000000
5,Tjoapack,All Tjoapack,2026-02-01,28,6931.0000,0,150.0,0.000000
6,Tjoapack,All Tjoapack,2025-01-01,17,5389.0000,0,0.0,0.000000
7,Tjoapack,All Tjoapack,2025-03-01,27,6665.0000,0,2123.0,0.000000
8,Tjoapack,All Tjoapack,2025-10-01,28,10504.0000,0,0.0,0.000000
9,Tjoapack,All Tjoapack,2025-12-01,14,4001.0000,0,0.0,0.000000
